In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> GPU, then re-run.'
print('GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/fno_results'                # finished results land here
DATA_CACHE = '/content/drive/MyDrive/fno_data/Darcy_421.zip'   # data cached here after first download
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(DATA_CACHE), exist_ok=True)
print('results ->', SAVE_DIR)

In [ ]:
import os
if not os.path.isdir('/content/neural-operator-reproduction'):
    !git clone https://github.com/gdimitrovdev/neural-operator-reproduction.git /content/neural-operator-reproduction
%cd /content/neural-operator-reproduction
!git pull -q
!pip install -q scipy pyyaml -U gdown

In [ ]:
import os, shutil
os.makedirs('data', exist_ok=True)
need = [f'data/piececonst_r421_N1024_smooth{i}.mat' for i in (1, 2)]
if not all(os.path.exists(f) for f in need):
    if os.path.exists(DATA_CACHE):
        print('copy cached zip from Drive')
        shutil.copy(DATA_CACHE, 'data/Darcy_421.zip')
    else:
        print('download from Google Drive')
        !gdown 1Z1uxG9R8AdAGJprG5STcphysjm56_0Jf -O data/Darcy_421.zip
        shutil.copy('data/Darcy_421.zip', DATA_CACHE)
    !cd data && unzip -o Darcy_421.zip && rm Darcy_421.zip
!ls -la data/

In [ ]:
# (model, config, epochs). LNO at s=85; GNO/MGNO at s=43, batch 1 (dense kernel OOMs otherwise).
MODELS = [# ('LNO', 'configs/darcy_lno2d.yaml', 200),  # done, see results/darcy_lno2d_s85
          ('GNO', 'configs/darcy_gno2d_s43.yaml', 100),
          ('MGNO', 'configs/darcy_mgno2d_s43.yaml', 100)]

import yaml, subprocess, glob, os, csv, shutil
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

def make_config(src, epochs):
    cfg = yaml.safe_load(open(src))
    cfg['training']['epochs'] = epochs
    tmp = os.path.join('configs', os.path.basename(src).replace('.yaml', f'_ep{epochs}.yaml'))
    yaml.safe_dump(cfg, open(tmp, 'w'))
    return tmp, cfg['logging']['experiment_name']

def run_and_save(name, src, epochs):
    exp = yaml.safe_load(open(src))['logging']['experiment_name']
    dest = os.path.join(SAVE_DIR, exp)
    saved = os.path.join(dest, 'same_resolution_eval.csv')
    if os.path.exists(saved):
        row = next(csv.DictReader(open(saved)))
        print(f"{name}: already on Drive (rel L2 = {float(row['test_rel_l2']):.4f} at s={row['resolution']}), skipping")
        return float(row['test_rel_l2']), int(row['resolution'])
    cfg_path, exp = make_config(src, epochs)
    print(f'{name}: training {epochs} epochs', flush=True)

    r = subprocess.run(['python', 'scripts/train_darcy_operator.py', '--config', cfg_path])
    if r.returncode != 0:
        print(f'{name} training failed (exit {r.returncode}), see traceback above', flush=True)
        return None, None
    run_dir = sorted(glob.glob(f'runs/{exp}_*'))[-1]
    subprocess.run(['python', 'scripts/evaluate.py', '--run', run_dir, '--mode', 'same'], check=True)
    os.makedirs(dest, exist_ok=True)
    for fn in ['metrics.csv', 'same_resolution_eval.csv', 'summary.json', 'config.yaml', 'best_model.pt']:
        p = os.path.join(run_dir, fn)
        if os.path.exists(p):
            shutil.copy(p, dest)
    row = next(csv.DictReader(open(os.path.join(run_dir, 'same_resolution_eval.csv'))))
    err, res = float(row['test_rel_l2']), int(row['resolution'])
    print(f'{name}: rel L2 = {err:.4f} at s={res}  (saved to {dest})', flush=True)
    return err, res

results = {}
for name, src, epochs in MODELS:
    results[name] = run_and_save(name, src, epochs)

In [ ]:
import os, csv, glob
paper = {'GNO': 0.0346, 'LNO': 0.0520, 'MGNO': 0.0416, 'FNO': 0.0108}  # all at s=85
rows = {'FNO': (0.0088, 85)}  # local reproduction (results/darcy_fno2d_s85)
for m in ['LNO', 'GNO', 'MGNO']:
    for c in sorted(glob.glob(os.path.join(SAVE_DIR, f'darcy_{m.lower()}2d*'))):
        p = os.path.join(c, 'same_resolution_eval.csv')
        if os.path.exists(p):
            r = next(csv.DictReader(open(p)))
            rows[m] = (float(r['test_rel_l2']), int(r['resolution']))

print(f"{'model':6}{'ours':>12}{'@s':>6}{'paper(s=85)':>14}")
for m in ['GNO', 'LNO', 'MGNO', 'FNO']:
    if m in rows:
        err, res = rows[m]
        print(f"{m:6}{err:>12.4f}{res:>6}{paper[m]:>14.4f}")
    else:
        print(f"{m:6}{'not run':>12}{'-':>6}{paper[m]:>14.4f}")